In [1]:
# 1. 导入必要的库
from vnpy.trader.setting import SETTINGS
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import  HistoryRequest
from vnpy.trader.datafeed import get_datafeed
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.database import DB_TZ
from vnpy.alpha import  logger
from datetime import datetime, timedelta
import rqdatac as rq
from pathlib import Path
from tqdm import tqdm
import json

In [2]:
# 2. 配置RQData数据服务

# 初始化数据服务
datafeed = get_datafeed()
print(f'数据服务类型: {datafeed.__class__.__name__}')

# 尝试初始化
inited = datafeed.init(output=print)
print(f'初始化结果: {inited}')

数据服务类型: RqdataDatafeed
初始化结果: True


In [54]:
a = rq.get_price('000001.XSHE', datetime(2025,5,8),datetime(2026,6,8), frequency='180d', adjust_type='none', skip_suspended=False)

In [55]:
print(a)

                          close  limit_down  prev_close  limit_up  \
order_book_id date                                                  
000001.XSHE   2026-01-27  10.94        9.86       10.96     12.06   
              2026-06-08  11.03        9.88       10.98     12.08   

                                volume  num_trades   open   high    low  \
order_book_id date                                                        
000001.XSHE   2026-01-27  2.013445e+10  12461827.0  11.00  13.33  10.92   
              2026-06-08  7.615059e+09   5088132.0  10.94  11.60  10.43   

                          total_turnover  
order_book_id date                        
000001.XSHE   2026-01-27    2.385498e+11  
              2026-06-08    8.381156e+10  


In [56]:
quota = rq.user.get_quota()
print(quota)

{'bytes_used': 10020820, 'bytes_limit': 209715200.0, 'remaining_days': 386, 'license_type': 'EDU'}


In [3]:
# 3. 路径配置

BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'
MINUTE_PATH = LAB_PATH / 'minute'
DAILY_PATH = LAB_PATH / 'daily'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [ ]:
# 4. 数据下载

vt_index_symbol = "000300.SSE"
rq_index_symbol = "000300.XSHG"

# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime(2026,5,8)
extended_days = 370
extended_years = 8
extended_start = start - timedelta(years=extended_days)
print(extended_start)

interval1 = Interval.MINUTE      #数据频率

# 之前的跨度
start2 = datetime(2018, 1, 1)
end2 = datetime(2026, 4, 23)

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = end

test_start2 = datetime(2025, 1, 1)
test_end2 = end2
interval2 = Interval.DAILY


trading_days = rq.get_trading_dates(  # 获取交易日
            start,
            end,
            market="cn"  # 'cn' 代表中国证券市场
        )
trading_days_str = [d.isoformat() for d in trading_days]
with open(lab.trading_days_path, "w+") as f:
    json.dump(trading_days_str, f, indent=2)


In [ ]:
# 4.1 下载成分股列表

# 下载总时间跨度的动态沪深300股票池股票代码
data = rq.index_components(rq_index_symbol, start_date=datetime(2010,1,1), end_date=end)

# 将rq的合约代码转化内vnpy的合约代码
vt_index_components = {}
for dt, rq_symbols in data.items():
    vt_symbols: list = []

    for rq_symbol in rq_symbols:
        vt_symbol = rq_symbol.replace("XSHG", "SSE").replace("XSHE", "SZSE")
        vt_symbols.append(vt_symbol)

    vt_index_components[dt.strftime("%Y-%m-%d")] = vt_symbols    #index_components = {"%Y-%m-%d":[成分股列表]，....}

# 保存
lab.save_component_data(vt_index_symbol, vt_index_components)

In [ ]:
# 加载成分股代码
component_symbols1 = lab.load_component_symbols(vt_index_symbol, start, end)
component_symbols2 = lab.load_component_symbols(vt_index_symbol, start2, end2)
print(len(component_symbols1))
print(len(component_symbols2))

In [ ]:
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)

In [ ]:
task_symbols = component_symbols

In [ ]:
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5/10000,
        short_rate=15/10000,
        size=1,
        pricetick=0.0001,
    )

In [ ]:
start = start.replace(tzinfo=DB_TZ)    # 下载要标明时区
end = end.replace(tzinfo=DB_TZ)
extend_start = extend_start.replace(tzinfo=DB_TZ)
start2 = start2.replace(tzinfo=DB_TZ)    # 下载要标明时区
end2 = end2.replace(tzinfo=DB_TZ)

In [ ]:
#最新的股票
task_symbols = [s for s in component_symbols1 if s not in component_symbols2]    # 目标股票
print(len(task_symbols))

In [4]:
vt_index_symbol = "000300.SSE"
rq_index_symbol = "000300.XSHG"

# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime(2026,5,30)
start = start.replace(tzinfo=DB_TZ)    # 下载要标明时区
end = end.replace(tzinfo=DB_TZ)

In [6]:
#--------------------下载新的股票的全时间跨度的k线(mink)

# 获取所有 .parquet 文件的名称部分（不含扩展名）
# done_symbols = [f.stem for f in Path(MINUTE_PATH).glob("*.parquet")]
# task_symbols = [vt_symbol for vt_symbol in task_symbols if vt_symbol not in done_symbols]
task_symbols = ['002450.SZSE', '002411.SZSE', '600074.SSE', '000413.SZSE', '600068.SSE', '000671.SZSE', '600485.SSE', '000961.SZSE', '600297.SSE', '000540.SZSE']
n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, Interval.DAILY, 'post')
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

  0%|          | 0/10 [00:00<?, ?it/s]

下载 002450.SZSE ...


  0%|          | 0/10 [00:00<?, ?it/s]


ShapeError: unable to append to a DataFrame of width 9 with a DataFrame of width 8

In [ ]:
#--------------------下载新的股票的全时间跨度的k线(mink)

# 获取所有 .parquet 文件的名称部分（不含扩展名）
# done_symbols = [f.stem for f in Path(MINUTE_PATH).glob("*.parquet")]
# task_symbols = [vt_symbol for vt_symbol in task_symbols if vt_symbol not in done_symbols]

n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval1)
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
#--------------------下载新的股票的全时间跨度的k线(日k)
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), datetime(2013,1,1).replace(tzinfo=DB_TZ), datetime(2017,1,1).replace(tzinfo=DB_TZ), interval2)
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
#--------------------下载新的股票的全时间跨度的k线(日k)
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval2)
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
#--------------------旧的股票只缺end2到end的k线
task_symbols = component_symbols2

In [ ]:
print(len(task_symbols))

In [ ]:
##--------------------下载旧的股票的剩下时间跨度的k线(mink)
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), end2, end, interval1)
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
##--------------------下载旧的股票的剩下时间跨度的k线(日k)
task_symbols = task_symbols + [vt_index_symbol]
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), end2, end , interval2)
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
quota = rq.user.get_quota()
print(quota)